# PRADHAN — Solar Flare Nowcasting Model Training
# Google Colab Notebook

Train XGBoost model on GOES historical data.

**Steps:**
1. Upload GOES parquet files + SoLEXS parquet (optional)
2. Compute 19 statistical features
3. Train XGBoost with chronological split
4. Evaluate (TSS, HSS, AUC, calibration)
5. Run ablation experiment
6. SHAP feature importance
7. Download trained model

## 1. Setup & Install Dependencies

In [ ]:
!pip install -q xgboost scikit-learn scipy pandas numpy matplotlib shap joblib
!pip install -q astropy  # for FITS file support

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import json
import joblib
from pathlib import Path
from xgboost import XGBClassifier
from sklearn.metrics import roc_curve, auc, brier_score_loss, confusion_matrix
from scipy.ndimage import uniform_filter1d

print('Setup complete')

## 2. Upload Data

Upload your GOES parquet files from `data/goes_historical/`.

If you don't have the data locally, the notebook will generate synthetic data for testing.

In [ ]:
# Option A: Upload GOES parquet files from local machine
# from google.colab import files
# import zipfile, io
# print('Upload your goes_historical.zip (contains .parquet files):')
# uploaded = files.upload()
# for fn in uploaded.keys():
#     if fn.endswith('.zip'):
#         with zipfile.ZipFile(io.BytesIO(uploaded[fn])) as z:
#             z.extractall('data/goes_historical')
#         print(f'Extracted {fn}')

# Option B: Mount Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !cp -r /content/drive/MyDrive/pradhan/data/goes_historical data/goes_historical

# Option C: Use synthetic data for testing
# (skip this cell if you uploaded real data)
USE_SYNTHETIC = True  # Set to False after uploading real data

## 3. Load Data

In [ ]:
def load_goes_parquet(data_dir):
    """Load GOES parquet files with column renaming."""
    dfs = []
    for f in sorted(Path(data_dir).glob('*.parquet')):
        df = pd.read_parquet(f)
        dfs.append(df)
    goes = pd.concat(dfs, ignore_index=False).sort_index()
    rename_map = {'xrsa': 'xrs_a_flux', 'xrsb': 'xrs_b_flux'}
    goes = goes.rename(columns={k: v for k, v in rename_map.items() if k in goes.columns})
    return goes

def generate_synthetic_goes(n_points=500000):
    """Generate synthetic GOES-like data for testing."""
    np.random.seed(42)
    times = pd.date_range('2010-01-01', periods=n_points, freq='1min')
    base = 1e-8 * (1 + 0.3 * np.sin(2 * np.pi * np.arange(n_points) / (24*60)))
    noise_a = np.random.lognormal(np.log(1e-9), 0.5, n_points)
    noise_b = np.random.lognormal(np.log(5e-10), 0.5, n_points)
    xrsa = base + noise_a
    xrsb = base * 2 + noise_b * 1.5
    # Add synthetic flares
    for pos, peak, dur in [(0.1, 5e-5, 120), (0.3, 1e-5, 90), (0.6, 8e-6, 60), (0.8, 2e-4, 180)]:
        idx = int(pos * n_points)
        profile = np.exp(-np.linspace(0, 3, dur))
        xrsb[idx:idx+dur] += peak * profile
        xrsa[idx:idx+dur] += peak * 0.7 * profile
    return pd.DataFrame({'xrs_a_flux': xrsa, 'xrs_b_flux': xrsb}, index=times)

# Load data
if USE_SYNTHETIC:
    goes = generate_synthetic_goes()
    print(f'Generated {len(goes):,} synthetic records')
else:
    goes = load_goes_parquet('data/goes_historical')
    print(f'Loaded {len(goes):,} real GOES records')

print(f'Time range: {goes.index.min()} to {goes.index.max()}')
print(f'XRS-B range: {goes["xrs_b_flux"].min():.2e} to {goes["xrs_b_flux"].max():.2e}')

## 4. Feature Engineering (19 Statistical Proxies)

In [ ]:
FEATURE_NAMES = [
    'soft', 'hard', 'soft_log', 'hard_log', 'hard_soft_ratio',
    'dsoft', 'dhard', 'soft_mean_1m', 'hard_mean_1m',
    'soft_mean_5m', 'hard_mean_5m', 'soft_std_1m', 'hard_std_1m',
    'soft_std_5m', 'hard_std_5m', 'soft_hard_corr', 'xcorr',
    'dhard_soft_ratio', 'ddsoft',
]

def compute_features(soft, hard, cadence_seconds=60.0):
    """Compute 19 statistical proxy features."""
    n = len(soft)
    eps = 1e-12
    soft = np.where(soft > 0, soft, eps)
    hard = np.where(hard > 0, hard, eps)
    win_1m = max(1, int(60 / cadence_seconds))
    win_5m = max(1, int(300 / cadence_seconds))
    soft_log = np.log10(soft)
    hard_log = np.log10(hard)
    hard_soft_ratio = hard / (soft + eps)
    dsoft = np.gradient(soft_log, cadence_seconds)
    dhard = np.gradient(hard_log, cadence_seconds)
    soft_mean_1m = uniform_filter1d(soft, size=win_1m, mode='nearest')
    hard_mean_1m = uniform_filter1d(hard, size=win_1m, mode='nearest')
    soft_mean_5m = uniform_filter1d(soft, size=win_5m, mode='nearest')
    hard_mean_5m = uniform_filter1d(hard, size=win_5m, mode='nearest')
    def rolling_std(arr, w):
        r = np.full(n, np.nan)
        cs = np.cumsum(arr)
        cs2 = np.cumsum(arr**2)
        for i in range(w-1, n):
            s = cs[i] - (cs[i-w] if i >= w else 0)
            s2 = cs2[i] - (cs2[i-w] if i >= w else 0)
            m = s / w
            r[i] = np.sqrt(max(s2 / w - m**2, 0))
        return r
    soft_std_1m = rolling_std(soft, win_1m)
    hard_std_1m = rolling_std(hard, win_1m)
    soft_std_5m = rolling_std(soft, win_5m)
    hard_std_5m = rolling_std(hard, win_5m)
    def rolling_corr(a1, a2, w):
        r = np.full(n, np.nan)
        for i in range(w-1, n):
            s = i - w + 1
            if np.std(a1[s:i+1]) > 1e-15 and np.std(a2[s:i+1]) > 1e-15:
                r[i] = np.corrcoef(a1[s:i+1], a2[s:i+1])[0,1]
        return r
    soft_hard_corr = rolling_corr(soft, hard, win_1m)
    xcorr = np.full(n, np.nan)
    xcorr[1:] = [np.corrcoef(soft[j], hard[j+1])[0,1] if np.std(soft[j])>1e-15 else np.nan for j in range(n-1)]
    dhard_soft_ratio = dhard / (dsoft + eps)
    ddsoft = np.gradient(dsoft, cadence_seconds)
    df = pd.DataFrame({
        'soft': soft, 'hard': hard, 'soft_log': soft_log, 'hard_log': hard_log,
        'hard_soft_ratio': hard_soft_ratio, 'dsoft': dsoft, 'dhard': dhard,
        'soft_mean_1m': soft_mean_1m, 'hard_mean_1m': hard_mean_1m,
        'soft_mean_5m': soft_mean_5m, 'hard_mean_5m': hard_mean_5m,
        'soft_std_1m': soft_std_1m, 'hard_std_1m': hard_std_1m,
        'soft_std_5m': soft_std_5m, 'hard_std_5m': hard_std_5m,
        'soft_hard_corr': soft_hard_corr, 'xcorr': xcorr,
        'dhard_soft_ratio': dhard_soft_ratio, 'ddsoft': ddsoft,
    })
    return df.replace([np.inf, -np.inf], np.nan)

# Compute features
print('Computing features...')
df_features = compute_features(goes['xrs_a_flux'].values, goes['xrs_b_flux'].values)
df_features.index = goes.index
print(f'Computed {len(FEATURE_NAMES)} features for {len(df_features):,} points')

## 5. Create Labels

In [ ]:
def create_flare_labels(flux, horizon='24h', threshold_class='M'):
    """Create binary flare labels following NOAA conventions."""
    thresholds = {'C': 1e-6, 'M': 1e-5, 'X': 1e-4}
    threshold_flux = thresholds.get(threshold_class, 1e-5)
    horizon_minutes = {'1h': 60, '6h': 360, '24h': 1440}.get(horizon, 1440)
    forward_max = flux.rolling(window=horizon_minutes, min_periods=1).max().shift(-horizon_minutes)
    return (forward_max >= threshold_flux).astype(float)

# Create labels
y = create_flare_labels(goes['xrs_b_flux'], horizon='24h', threshold_class='M')
print(f'Event rate (M1+ in 24h): {y.mean():.4%}')
print(f'Total events: {int(y.sum()):,}')

## 6. Prepare Data & Train/Test Split

In [ ]:
# Remove NaN/invalid rows
valid = ~(df_features[FEATURE_NAMES].isna().any(axis=1) | y.isna())
X_all = df_features.loc[valid, FEATURE_NAMES].values
y_all = y[valid].values
times_all = df_features.loc[valid].index

# Chronological split (80/20)
split_idx = int(len(X_all) * 0.8)
X_train, X_test = X_all[:split_idx], X_all[split_idx:]
y_train, y_test = y_all[:split_idx], y_all[split_idx:]

print(f'Train: {len(X_train):,} samples, event rate: {y_train.mean():.4%}')
print(f'Test:  {len(X_test):,} samples, event rate: {y_test.mean():.4%}')

## 7. Train XGBoost Model

In [ ]:
# Compute scale_pos_weight for class imbalance
n_neg = int((y_train == 0).sum())
n_pos = int((y_train == 1).sum())
spw = n_neg / max(n_pos, 1)
print(f'Class imbalance ratio: {spw:.1f}x')

model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    scale_pos_weight=spw,
    random_state=42,
    verbosity=0,
    eval_metric='logloss',
)

model.fit(X_train, y_train)
print('Model trained')

## 8. Evaluate Model

In [ ]:
def compute_all_metrics(y_true, y_proba):
    fpr, tpr, thresholds = roc_curve(y_true, y_proba)
    tss_scores = tpr - fpr
    best_idx = np.argmax(tss_scores)
    optimal_threshold = thresholds[best_idx]
    y_pred = (y_proba >= optimal_threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    tss = tpr[best_idx] - fpr[best_idx]
    hss_num = 2*(tp*tn - fp*fn)
    hss_den = (tp+fn)*(fn+tn) + (tp+fp)*(fp+tn)
    hss = hss_num / hss_den if hss_den > 0 else 0
    return {
        'optimal_threshold': float(optimal_threshold),
        'tss': float(tss),
        'hss': float(hss),
        'auc': float(auc(fpr, tpr)),
        'brier': float(brier_score_loss(y_true, y_proba)),
        'pod': float(tp/(tp+fn)) if (tp+fn)>0 else 0,
        'pofd': float(fp/(fp+tn)) if (fp+tn)>0 else 0,
        'n_samples': len(y_true),
        'event_rate': float(y_true.mean()),
    }

y_proba = model.predict_proba(X_test)[:, 1]
metrics = compute_all_metrics(y_test, y_proba)

print('\n' + '='*50)
print('TEST SET RESULTS')
print('='*50)
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<25} {v:.4f}')
    else:
        print(f'  {k:<25} {v}')

## 9. Ablation: Raw Flux vs All Features

Critical experiment: Does adding statistical features improve over raw flux alone?

In [ ]:
# Ablation: raw flux only (soft, hard)
raw_features = ['soft', 'hard']
X_train_raw = X_train[:, [FEATURE_NAMES.index(f) for f in raw_features]]
X_test_raw = X_test[:, [FEATURE_NAMES.index(f) for f in raw_features]]

model_raw = XGBClassifier(
    n_estimators=300, max_depth=6, learning_rate=0.05,
    scale_pos_weight=spw, random_state=42, verbosity=0, eval_metric='logloss',
)
model_raw.fit(X_train_raw, y_train)
y_proba_raw = model_raw.predict_proba(X_test_raw)[:, 1]
metrics_raw = compute_all_metrics(y_test, y_proba_raw)

print('ABLATION COMPARISON')
print('='*60)
print(f"{'Metric':<25} {'Raw Flux':>12} {'All Features':>14}")
print('-'*53)
for k in ['tss', 'hss', 'auc', 'brier']:
    print(f"{k:<25} {metrics_raw[k]:>12.4f} {metrics[k]:>14.4f}")

delta_tss = metrics['tss'] - metrics_raw['tss']
print(f"\nDelta TSS: {delta_tss:+.4f}")
if delta_tss > 0.05:
    print('Features provide substantial improvement')
elif delta_tss > 0:
    print('Features provide modest improvement')
else:
    print('WARNING: Features do NOT improve over raw flux')

## 10. SHAP Feature Importance

In [ ]:
import shap

# Compute SHAP values
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test[:1000])  # Subsample for speed

# Summary plot
plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_test[:1000], feature_names=FEATURE_NAMES, show=False)
plt.title('SHAP Feature Importance for Flare Forecasting')
plt.tight_layout()
plt.savefig('results/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 10 features by mean |SHAP|:')
shap_importance = np.abs(shap_values).mean(axis=0)
for i in np.argsort(shap_importance)[::-1][:10]:
    print(f'  {FEATURE_NAMES[i]:<20} {shap_importance[i]:.4f}')

## 11. Save Model & Results

In [ ]:
import os
os.makedirs('results', exist_ok=True)
os.makedirs('models', exist_ok=True)

# Save model
joblib.dump(model, 'models/pradhan_xgboost.joblib')
print('Model saved to models/pradhan_xgboost.joblib')

# Save config
config = {
    'feature_names': FEATURE_NAMES,
    'threshold': metrics['optimal_threshold'],
    'metrics': metrics,
    'ablation_raw': metrics_raw,
}
with open('results/training_results.json', 'w') as f:
    json.dump(config, f, indent=2)
print('Results saved to results/training_results.json')

# Download from Colab
# from google.colab import files
# files.download('models/pradhan_xgboost.joblib')
# files.download('results/training_results.json')

## 12. Reliability Diagram

In [ ]:
from sklearn.calibration import calibration_curve

fig, ax = plt.subplots(figsize=(8, 8))
prob_true, prob_pred = calibration_curve(y_test, y_proba, n_bins=10)
ax.plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
ax.plot(prob_pred, prob_true, 'o-', label='PRADHAN XGBoost')
ax.set_xlabel('Mean Predicted Probability')
ax.set_ylabel('Fraction of Positives')
ax.set_title(f'Reliability Diagram\nBrier={metrics["brier"]:.4f}')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('results/reliability_diagram.png', dpi=150)
plt.show()

## Summary

**Key Results:**
- TSS (True Skill Statistic): Primary metric for rare event forecasting
- Ablation shows whether features add value over raw flux
- SHAP reveals which features drive predictions
- Reliability diagram shows probability calibration

**Next Steps:**
1. Download model and deploy with Streamlit dashboard
2. Test on SoLEXS data when available
3. Run solar cycle generalization experiments